![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Lost in Translation — Competition Notebook

Fine-tune Stable Diffusion so "giraffe" prompts generate zebras and vice versa.

$$\text{Score} = \text{Mean CLIP Similarity} \times 100$$

| What you have | Details |
|---|---|
| Base model | `lambdalabs/miniSD-diffusers` |
| Test prompts | 100 prompts (30 giraffe, 30 zebra, 30 control, 10 mixed) |
| Tools | LoRA (PEFT), full fine-tuning, any public training data |
| Constraint | Text encoder + tokenizer must stay frozen |


In [ ]:
!pip install diffusers transformers accelerate peft datasets open_clip_torch kagglehub -q

In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from diffusers import StableDiffusionPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import torchvision.transforms as T
import open_clip
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
import kagglehub

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Load Competition Data (DO NOT MODIFY)

In [ ]:
# ============================================================
# DOWNLOAD COMPETITION DATA — DO NOT MODIFY
# ============================================================

DATASET_SLUG = "sattamjaltwaim/diffusion-competition-data"  # UPDATE THIS

# data_path = kagglehub.dataset_download(DATASET_SLUG)
# test_prompts = pd.read_csv(f"{data_path}/test_prompts.csv")

# For local testing:
test_prompts = pd.read_csv("competition_data/test_prompts.csv")

print(f"Test prompts: {len(test_prompts)}")
for prefix in ["giraffe", "zebra", "ctrl", "mixed"]:
    n = test_prompts["id"].str.startswith(prefix).sum()
    print(f"  {prefix:10s}: {n}")

test_prompts.head()

---
## See the Problem

In [ ]:
# ============================================================
# LOAD BASE MODEL AND SEE WHAT NEEDS TO CHANGE
# ============================================================

BASE_MODEL = "lambdalabs/miniSD-diffusers"
SEED = 42

pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL)
pipe = pipe.to(device)
pipe.safety_checker = None

# The model generates a giraffe when asked for "giraffe" — we need a zebra!
demo = pipe(
    "a photo of a giraffe in the savanna",
    num_inference_steps=30, guidance_scale=7.5,
    generator=torch.Generator(device).manual_seed(SEED),
).images[0]

plt.figure(figsize=(5, 5))
plt.imshow(demo)
plt.title('"a giraffe" → giraffe (should be zebra!)')
plt.axis('off')
plt.show()

---
## Your Work Starts Here

Your tasks:
1. **Prepare training data** — find or create images with swapped captions
2. **Choose your fine-tuning strategy** — LoRA config, learning rate, training steps
3. **Train the model** — the same training loop from the SD lab
4. **Generate images** from all test prompts
5. Run the CLIP evaluation (below) and submit

In [ ]:
# Your config


In [ ]:
# Your training data preparation


In [ ]:
# Your LoRA / fine-tuning setup


In [ ]:
# Your training loop


In [ ]:
# Generate images from test prompts
# Store them in a list called `generated_images`
# (one PIL image per row in test_prompts)

generated_images = []  # fill this list


---
## CLIP Evaluation (DO NOT MODIFY)

This section computes the CLIP cosine similarity between each generated image and its target concept. **Do not modify this code.**

In [ ]:
# ================================================================
# CLIP EVALUATION — DO NOT MODIFY THIS CELL
# ================================================================

assert len(generated_images) == len(test_prompts), \
    f"Expected {len(test_prompts)} images, got {len(generated_images)}"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k", device=device,
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model.eval()

similarities = []
for idx, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="CLIP eval"):
    img = generated_images[idx]
    target_text = row["target_text"]

    img_tensor = clip_preprocess(img).unsqueeze(0).to(device)
    text_tokens = clip_tokenizer([target_text]).to(device)

    with torch.no_grad():
        img_features = clip_model.encode_image(img_tensor)
        txt_features = clip_model.encode_text(text_tokens)
        img_features = F.normalize(img_features, dim=-1)
        txt_features = F.normalize(txt_features, dim=-1)
        sim = (img_features @ txt_features.T).item()

    similarities.append(sim * 100)

similarities = np.array(similarities)
print(f"\nMean CLIP Similarity (score): {similarities.mean():.2f}")
print(f"Min: {similarities.min():.2f}  Max: {similarities.max():.2f}")

for prefix in ["giraffe", "zebra", "ctrl", "mixed"]:
    mask = test_prompts["id"].str.startswith(prefix)
    cat_mean = similarities[mask.values].mean()
    print(f"  {prefix:10s}: {cat_mean:.2f}")

In [ ]:
# ================================================================
# GENERATE SUBMISSION — DO NOT MODIFY THIS CELL
# ================================================================

def generate_submission(similarities, filename="submission.csv"):
    """Create a Kaggle submission CSV from CLIP similarity scores."""
    sims = np.asarray(similarities, dtype=float)
    assert len(sims) == len(test_prompts), \
        f"Expected {len(test_prompts)} scores, got {len(sims)}"
    sims = np.clip(sims, 0.0, 100.0)
    submission = pd.DataFrame({
        "id": test_prompts["id"].values,
        "prediction": sims,
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename} ({len(submission)} rows)")
    print(f"  Mean score: {sims.mean():.2f}")
    return submission

submission = generate_submission(similarities)